# Figure 2 | Tissue-specific expression of novel lncRNAs

**a.** In each gene biotype category, the percentage of genes expressed versus intergenic controls at FDR 5%. Each point is a tissue; outlined bars are genes expressed in any tissue or in pooled data. 95.5% of novel lncRNAs are detected in at least one tissue.

**b.** Cumulative tissue-specificity index (τ) distributions for each biotype. Dotted lines mark housekeeping (τ = 0.15) and tissue-specific (τ = 0.85) cutoffs. Novel lncRNAs are more likely to be tissue-specific than the other biotypes. Inset: expression-adjusted odds of being tissue-specific (τ > 0.85) for existing lncRNAs, protein-coding genes, and pseudogenes versus novel lncRNAs, from logistic regression on tissue-specific status with novel lncRNA as the reference category and including expression as a covariate. Expression is included as log10(TPM + 1) where TPM is the maximum median tissue TPM. Points, logistic-regression odds ratios (ORs); error bars, 95% CIs; asterisks, two-sided Wald p (* p < 0.05; ** p < 0.01; *** p < 0.001).

**c.** Counts of tissue-specific genes (τ > 0.85) in each biotype. Stacked bars are colored by the organ with the highest median expression. Testis-peak genes that remain tissue-specific after excluding testis are reassigned to their non-testis highest-expressed organ; genes that are tissue-specific only because of testis remain assigned to testis.

**d.** Counts of tissue-specific novel and existing lncRNAs whose macaque and/or marmoset ortholog is expressed at FDR 5% vs intergenic control and peaks in the same organ as in human (tissue-concordant). Paired bars are novel versus existing lncRNAs; stacked bars are colored by organ. lncRNAs with this conserved peak organ across primates are tissue-concordant lncRNAs.

**e.** Sample-level TPM of brain-specific novel lncRNA ENSG00000296434 (τ = 0.935) across tissue sites in GTEx, dGTEx, and NHP-dGTEx (macaque and marmoset). Box plots show median and interquartile range (IQR); whiskers, 1.5× IQR; points, outlier samples clipped at TPM = 6. Colors, tissue site/organ. Inset shows genomic context. Highest median expression is brain cerebellum in GTEx and brain sites in dGTEx / macaque / marmoset.


## Setup


In [ ]:
import gzip
import json
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import seaborn as sns
import statsmodels.formula.api as smf
from matplotlib.patches import Patch
from matplotlib.transforms import blended_transform_factory

plt.rc("axes", labelsize=6)
plt.rc("xtick", labelsize=6)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=5)
plt.rc("legend", title_fontsize=6)
plt.rc("font", size=6)
plt.rc("figure", titlesize=7)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.figsize"] = (7.2, 5.0)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

# Tunable params
fig_width, fig_height, fig_dpi = 7.2, 5.2, 300
tau_housekeeping = 0.15
tau_tissue_specific = 0.85
expected_n_bulk_tissues = 54  # histology subtypes excluded; includes 2 cell lines
expected_n_tau_tissues = 52   # bulk set minus cell lines

# Histology subtypes dropped from the official median TPM GCT (→ 54 bulk sites); cell lines also dropped for τ (→ 52)
histology_subtypes = {
    "Colon - Transverse - Muscularis",
    "Stomach - Mixed Cell",
    "Pancreas - Mixed Cell",
    "Small Intestine - Terminal Ileum - Mixed Cell",
    "Stomach - Mucosa",
    "Stomach - Muscularis",
    "Pancreas - Acini",
    "Small Intestine - Terminal Ileum - Lymphoid Aggregate",
    "Colon - Transverse - Mixed Cell",
    "Colon - Transverse - Mucosa",
    "Liver - Mixed Cell",
    "Pancreas - Islets",
    "Liver - Hepatocyte",
    "Liver - Portal Tract",
}
cell_line_tissues = {
    "Cells - EBV-transformed lymphocytes",
    "Cells - Cultured fibroblasts",
}
example_gene = "ENSG00000296434"
example_min_samples_per_site = 3
example_flank_bp = 40_000
example_neighbor_show_bp = 60_000
example_max_neighbor_gap_bp = 50_000

gene_type_order_split = ["novel_lncRNA", "existing_lncRNA", "protein_coding", "pseudogene"]
bar_gene_type_order = ["novel_lncRNA", "existing_lncRNA", "protein_coding", "pseudogene"]
lnc_split_order = ["novel_lncRNA", "existing_lncRNA"]
tau_gene_type_order = ["novel_lncRNA", "existing_lncRNA", "protein_coding", "pseudogene"][::-1]

gene_type_pretty = {
    "novel_lncRNA": "novel lncRNA",
    "existing_lncRNA": "existing lncRNA",
    "protein_coding": "protein-coding",
    "pseudogene": "pseudogene",
    "sRNA": "sRNA",
    "lncRNA": "lncRNA",
}
gene_type_palette = {
    "lncRNA": sns.color_palette("Paired", 12)[5],
    "novel_lncRNA": sns.color_palette("Paired", 12)[5],
    "existing_lncRNA": sns.color_palette("Paired", 12)[4],
    "protein_coding": sns.color_palette("Paired", 12)[1],
    "pseudogene": sns.color_palette("Paired", 12)[3],
    "sRNA": sns.color_palette("Paired", 12)[7],
}

## Paths

In [ ]:
gtex_colors_json = Path("refs/colors.json")
tissue_names_file = Path("annotation/refs/tissue_name.txt")
gencode_class = Path("path/to/gencode_v39_v47_gene_classification.txt")
lnc_category_file = Path("path/to/lncrna_category.txt")
gencode_annot_feather = Path("path/to/gencode.v47.annotation.gtf.feather")
gtex_sample_meta_file = Path("path/to/GTEx_Analysis_v10_Annotations_SampleAttributesDS.txt")

gtex_pooled_null = Path("path/to/GTEx.null_quantifications.txt")
gtex_median_tpm = Path("path/to/GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_median_tpm.gct.gz")
# Wide gene×tissue FDR matrix is rebuilt below from these per-tissue tables (Zenodo expression/).
gtex_by_tissue_null_dir = Path("path/to/by_tissue/GTEx")

dgtex_colors_csv = Path("path/to/dgtex_colors_final.csv")
dgtex_meta_file = Path("path/to/dGTEx_Analysis_2025-07-21_v1_RNA_Metadata_subset_EXCLUSIONS.xlsx")
macaque_meta_file = Path("path/to/NHP-dGTEx_Macaque_Analysis_2025-07-21_v1_RNA_Metadata_subset.xlsx")
marmoset_meta_file = Path("path/to/NHP-dGTEx_Marmoset_Analysis_2025-08-08_v1_RNA_Metadata_subset.xlsx")
orth_tsv = Path("path/to/dGTEx_nhpGTEx_liftedlncRNAs.tsv")

tpm_gtex = Path("path/to/GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_tpm.parquet")
tpm_dgtex = Path("path/to/dGTEx_Analysis_2025-12-10_v1_unstranded_collapsed_trimmed_RNASeQCv2.4.3_gene_tpm.parquet")
tpm_macaque = Path("path/to/NHP-dGTEx_Macaque_Analysis_2025-11-20_v1_unstranded_collapsed_RNASeQCv2.4.3_gene_tpm.parquet")
tpm_marmoset = Path("path/to/NHP-dGTEx_Marmoset_Analysis_2025-11-20_v1_unstranded_collapsed_RNASeQCv2.4.3_gene_tpm.parquet")

fig_dir = Path("figures")
fig_pdf = fig_dir / "figure_2.pdf"
fig_png = fig_dir / "figure_2.png"


## Load data


### Tissue colors

In [ ]:
with open(gtex_colors_json) as f:
    _colors = json.load(f)
    tissue_df = pd.DataFrame.from_dict(_colors, orient="index")
tissue_df = tissue_df.reset_index(names="tissue_name")
tissue_df["tissue_color_hex"] = "#" + tissue_df["tissue_color_hex"]
tissue_df["tissue_file_name"] = (
    tissue_df["tissue_name"]
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace(" - ", "_")
    .str.replace(" ", "_")
)
tissue_palette = dict(zip(tissue_df["tissue_name"], tissue_df["tissue_color_hex"]))
print(f"GTEx tissue colors: {len(tissue_df):,}")


### Gencode annotations


In [ ]:
gencode_genes = pd.read_table(gencode_class, low_memory=False)
gencode_genes = gencode_genes[gencode_genes["gene_type_grouped"] != "artifact"]
print(f"Gencode genes: {len(gencode_genes):,}")


### GTEx pooled null quantifications


In [ ]:
gtex_null = pd.read_table(gtex_pooled_null)
print(f"GTEx null genes: {len(gtex_null):,}")

### GTEx sample metadata

In [ ]:
gtex_sample_meta = pd.read_csv(gtex_sample_meta_file, sep="\t", low_memory=False)
print(f"GTEx sample metadata: {len(gtex_sample_meta):,} rows")

### Per-tissue expression calls (FDR 5%)

In [ ]:
# Rebuild gene × tissue FDR 5% matrix from Zenodo per-tissue null tables (panel a: 54 bulk sites).
_tissue_map = pd.read_csv(tissue_names_file, sep="\t")
_stem_to_pretty = dict(
    zip(_tissue_map["tissue_stem"].astype(str), _tissue_map["tissue_name"].astype(str))
)
_parts = []
for path in sorted(gtex_by_tissue_null_dir.glob("*.null_quantifications.txt")):
    stem = path.name.replace(".null_quantifications.txt", "")
    pretty = _stem_to_pretty.get(stem)
    if pretty is None:
        raise KeyError(f"No tissue_name for stem {stem!r} in {tissue_names_file}")
    if pretty in histology_subtypes:
        continue
    part = (
        pd.read_table(path, usecols=["gene_id", "fdr05_quantified"])
        .rename(columns={"fdr05_quantified": pretty})
        .set_index("gene_id")
        .astype(bool)
    )
    _parts.append(part)
if not _parts:
    raise FileNotFoundError(f"No per-tissue null tables in {gtex_by_tissue_null_dir}")
tissue_fdr = _parts[0]
for part in _parts[1:]:
    tissue_fdr = tissue_fdr.join(part, how="outer")
tissue_fdr = tissue_fdr.fillna(False).astype(bool)
if tissue_fdr.shape[1] != expected_n_bulk_tissues:
    raise ValueError(
        f"Expected {expected_n_bulk_tissues} bulk tissues after dropping histology subtypes, "
        f"found {tissue_fdr.shape[1]}: {list(tissue_fdr.columns)}"
    )
tissue_names = list(tissue_fdr.columns)
tissue_fdr_cols = [f"{t}.fdr05_quantified" for t in tissue_names]
fdr_for_merge = tissue_fdr.rename(columns=dict(zip(tissue_names, tissue_fdr_cols))).reset_index()
if fdr_for_merge.columns[0] != "gene_id":
    fdr_for_merge = fdr_for_merge.rename(columns={fdr_for_merge.columns[0]: "gene_id"})
null_expression_tissues = pd.merge(
    gencode_genes[["gene_id_v47", "gene_type_grouped", "change_level", "in_v47_full"]],
    fdr_for_merge,
    how="left",
    left_on="gene_id_v47",
    right_on="gene_id",
).rename(columns={"gene_id_v47": "gene_id"})
for c in tissue_fdr_cols:
    null_expression_tissues[c] = null_expression_tissues[c].fillna(False).astype(bool)
print(
    f"Built tissue FDR table: {len(null_expression_tissues):,} genes × {len(tissue_fdr_cols)} bulk sites "
    f"(panel a; cell lines included) from {gtex_by_tissue_null_dir}"
)

### Tissue median TPM (official GTEx v11 release)


In [ ]:
def load_gtex_official_median_tpm(path: Path) -> pd.DataFrame:
    """Load GTEx v11 official gene median TPM GCT; columns = pretty SMTSD names."""
    with gzip.open(path, "rt") as f:
        f.readline()
        f.readline()
        raw = pd.read_csv(f, sep="\t")
    med = raw.set_index("Name").drop(columns=["Description"], errors="ignore")
    tissue_map = pd.read_csv(tissue_names_file, sep="\t")
    stem_to_pretty = dict(zip(tissue_map["tissue_stem"], tissue_map["tissue_name"]))
    renamed = {}
    for c in med.columns:
        pretty = stem_to_pretty.get(c) or stem_to_pretty.get(c.replace("-", "_"))
        if pretty is None:
            raise KeyError(f"Cannot map official median tissue column {c!r}")
        renamed[c] = pretty
    return med.rename(columns=renamed)


median_tpm = load_gtex_official_median_tpm(gtex_median_tpm)
_before = median_tpm.shape[1]
median_tpm_bulk = median_tpm[[c for c in median_tpm.columns if c not in histology_subtypes]]
if median_tpm_bulk.shape[1] != expected_n_bulk_tissues:
    raise ValueError(
        f"Expected {expected_n_bulk_tissues} bulk tissues after dropping histology subtypes, "
        f"found {median_tpm_bulk.shape[1]}"
    )
median_tpm = median_tpm_bulk[[c for c in median_tpm_bulk.columns if c not in cell_line_tissues]]
print(
    f"Official median TPM: {_before} GCT columns → {median_tpm_bulk.shape[1]} bulk "
    f"(subtypes out) → {median_tpm.shape[1]} τ (cell lines out)"
)
if median_tpm.shape[1] != expected_n_tau_tissues:
    raise ValueError(
        f"Expected {expected_n_tau_tissues} τ tissues in median TPM, found {median_tpm.shape[1]}"
    )

### NHP-dGTEx lncRNA subset


In [ ]:
lnc_meta = pd.read_table(
    lnc_category_file,
    usecols=[
        "gene_id", "gene_name",
        "is_tissue_specific", "peak_tissue", "rescued_from_testis",
        "macaque_ortholog", "marmoset_ortholog",
        "macaque_tissue_concordant", "marmoset_tissue_concordant",
        "expressed_gtex", "expressed_dgtex", "expressed_macaque", "expressed_marmoset",
    ],
)
# tissue-concordant flags from lncrna_category.txt (organ/Uberon peak match + FDR 5% NHP expression)
lncrna_names = lnc_meta.set_index("gene_id")["gene_name"]
print(
    f"lncRNAs: {len(lnc_meta):,}; TS={int(lnc_meta['is_tissue_specific'].sum()):,}; "
    f"rescued={int(lnc_meta['rescued_from_testis'].sum()):,}; "
    f"macaque_tissue_concordant={int(lnc_meta['macaque_tissue_concordant'].sum()):,} "
    f"(organ match + FDR5% expressed); "
    f"marmoset_tissue_concordant={int(lnc_meta['marmoset_tissue_concordant'].sum()):,} "
    f"(organ match + FDR5% expressed)"
)


### dGTEx tissue colors


In [ ]:
dgtex_tissue_df = pd.read_csv(dgtex_colors_csv)
dgtex_tissue_palette = dict(zip(dgtex_tissue_df["Tissue Site Detail"], dgtex_tissue_df["Tissue Site Detail colorHex"]))
dgtex_tissue_group_palette = (
    dgtex_tissue_df.drop_duplicates("Tissue Site")
    .set_index("Tissue Site")["Tissue Site colorHex"]
    .to_dict()
)
dgtex_tissue_site_abbrv = (
    dgtex_tissue_df.sort_values("Tissue Site Detail")
    .drop_duplicates("Tissue Site")
    .set_index("Tissue Site")["Tissue Site Detail Abbr"]
    .to_dict()
)
dgtex_tissue_abbrv = dict(zip(dgtex_tissue_df["Tissue Site Detail"], dgtex_tissue_df["Tissue Site Detail Abbr"]))
dgtex_tissue_site_abbrv["Trachea"] = dgtex_tissue_abbrv.get("Trachea", "TRACHEA")
print(f"dGTEx tissue colors: {len(dgtex_tissue_palette):,}")


### dGTEx RNA metadata

In [ ]:
dgtex_meta = pd.read_excel(dgtex_meta_file)
print(f"dGTEx metadata: {len(dgtex_meta):,} rows")

### Macaque RNA metadata

In [ ]:
macaque_meta = pd.read_excel(macaque_meta_file)
print(f"macaque metadata: {len(macaque_meta):,} rows")

### Marmoset RNA metadata

In [ ]:
marmoset_meta = pd.read_excel(marmoset_meta_file)
print(f"marmoset metadata: {len(marmoset_meta):,} rows")

### Tissue-site lookups

In [ ]:
gtex_detail_to_site = gtex_sample_meta.drop_duplicates("SMTSD").set_index("SMTSD")["SMTS"].to_dict()
gtex_tissue_site_abbrv = (
    gtex_sample_meta.merge(tissue_df[["tissue_name", "tissue_abbrv"]], left_on="SMTSD", right_on="tissue_name")
    .drop_duplicates(["SMTS", "tissue_abbrv"])
    .sort_values("SMTSD")
    .drop_duplicates("SMTS")
    .set_index("SMTS")["tissue_abbrv"]
    .to_dict()
)
gtex_tissue_group_palette = (
    gtex_sample_meta.merge(tissue_df[["tissue_name", "tissue_color_hex"]], left_on="SMTSD", right_on="tissue_name")
    .drop_duplicates(["SMTS", "tissue_color_hex"])
    .dropna(subset=["tissue_color_hex"])
    .groupby("SMTS")["tissue_color_hex"]
    .first()
    .to_dict()
)
dgtex_detail_to_site = (
    dgtex_meta.drop_duplicates("Tissue Site Detail")
    .set_index("Tissue Site Detail")["Tissue Site"]
    .to_dict()
)
macaque_detail_to_site = (
    macaque_meta.drop_duplicates("Tissue Site Detail")
    .set_index("Tissue Site Detail")["Tissue Site"]
    .to_dict()
)
marmoset_detail_to_site = (
    marmoset_meta.drop_duplicates("Tissue Site Detail")
    .set_index("Tissue Site Detail")["Tissue Site"]
    .to_dict()
)

site_abbrv_by_dataset = {
    "GTEx": gtex_tissue_site_abbrv,
    "dGTEx": dgtex_tissue_site_abbrv,
    "Macaque": dgtex_tissue_site_abbrv,
    "Marmoset": dgtex_tissue_site_abbrv,
}
site_palette_by_dataset = {
    "GTEx": gtex_tissue_group_palette,
    "dGTEx": dgtex_tissue_group_palette,
    "Macaque": dgtex_tissue_group_palette,
    "Marmoset": dgtex_tissue_group_palette,
}
detail_to_site_by_dataset = {
    "GTEx": gtex_detail_to_site,
    "dGTEx": dgtex_detail_to_site,
    "Macaque": macaque_detail_to_site,
    "Marmoset": marmoset_detail_to_site,
}

### Human-NHP ortholog table

In [ ]:
orth = pd.read_csv(orth_tsv, sep="\t")
for col in ["gene_id.macaque", "gene_id.marmoset"]:
    orth[col] = orth[col].replace(["NA", "-"], None)
orth = orth.set_index("gene_id.human")
print(f"Ortholog table: {len(orth):,}")

### GTEx v11 gene TPM

In [ ]:
gtex_tpm = pd.read_parquet(tpm_gtex)
print(f"GTEx TPM: {gtex_tpm.shape[0]:,} genes × {gtex_tpm.shape[1]:,} columns")

### dGTEx gene TPM

In [ ]:
dgtex_tpm = pd.read_parquet(tpm_dgtex)
print(f"dGTEx TPM: {dgtex_tpm.shape[0]:,} genes × {dgtex_tpm.shape[1]:,} columns")

### Macaque gene TPM

In [ ]:
macaque_tpm = pd.read_parquet(tpm_macaque)
print(f"macaque TPM: {macaque_tpm.shape[0]:,} genes × {macaque_tpm.shape[1]:,} columns")

### Marmoset gene TPM

In [ ]:
marmoset_tpm = pd.read_parquet(tpm_marmoset)
print(f"marmoset TPM: {marmoset_tpm.shape[0]:,} genes × {marmoset_tpm.shape[1]:,} columns")

### GENCODE v47 annotation

In [ ]:
gencode_annot = pa.ipc.open_file(gencode_annot_feather).read_all()
print(f"GENCODE v47 annotation: {gencode_annot.num_rows:,} features")

## Run analysis


### Expression breadth summary


In [ ]:
# Pooled FDR 5% calls; bar height = pooled ∪ any of 54 bulk sites (cell lines included)
pooled_expr = gtex_null.set_index("gene_id")["GTEx.fdr05_quantified"]

_expr_rows, _hollow_rows = [], []
expr_any_counts = {}
for gt in gene_type_order_split:
    sel = null_expression_tissues["gene_type_grouped"].eq(gt) & null_expression_tissues["in_v47_full"]
    sub = null_expression_tissues.loc[sel]
    n = len(sub)
    for tcol in tissue_fdr_cols:
        tissue = tcol.replace(".fdr05_quantified", "")
        pct = 100 * sub[tcol].fillna(False).sum() / n if n > 0 else 0
        _expr_rows.append((gt, tissue, pct))
    any_tissue = sub[tissue_fdr_cols].fillna(False).sum(axis=1) > 0
    # merge can leave duplicate gene_id cols; take the first
    _gid = sub["gene_id"]
    if isinstance(_gid, pd.DataFrame):
        _gid = _gid.iloc[:, 0]
    pooled = _gid.map(pooled_expr).fillna(False).astype(bool)
    any_expr = any_tissue.to_numpy() | pooled.to_numpy()
    pct_any = 100 * any_expr.sum() / n if n > 0 else 0
    _hollow_rows.append((gt, pct_any, gene_type_order_split.index(gt)))
    expr_any_counts[gt] = (int(any_expr.sum()), int(n))
tissue_summary = pd.DataFrame(_expr_rows, columns=["gene_type_grouped", "tissue", "pct"])
print(
    f"Expression breadth summary ready "
    f"({tissue_summary['tissue'].nunique()} bulk sites as dots; bars = pooled ∪ any of those)"
)


### Tau and tissue specificity


In [ ]:
def get_tau_category(tau_val):
    edges = [0, tau_housekeeping, tau_tissue_specific, 1]
    labels = ["Housekeeping", "Midrange", "Tissue specific"]
    for i in range(len(edges) - 1):
        if edges[i] <= tau_val < edges[i + 1] or (i == len(edges) - 2 and tau_val == edges[-1]):
            return labels[i]
    return np.nan

def compute_tau(tpm_df):
    log_tpm = np.log10(tpm_df + 1)
    max_expr = log_tpm.max(axis=1)
    normed = log_tpm.div(max_expr, axis=0)
    n_tissues = log_tpm.shape[1]
    tau = (1 - normed).sum(axis=1) / (n_tissues - 1)
    tau[max_expr == 0] = np.nan
    tau.name = "tau"
    return tau

expressed_gene_ids = set(gtex_null[gtex_null["GTEx.fdr05_quantified"]]["gene_id"])

median_tpm_filtered = median_tpm.loc[median_tpm.index.isin(expressed_gene_ids)]
tissue_map = gtex_sample_meta.set_index("SMTSD")["SMTS"].to_dict()
tissue_to_organ_map = {t: tissue_map.get(t, t) for t in median_tpm.columns}
non_testis_cols = [c for c in median_tpm.columns if tissue_to_organ_map.get(c, c) != "Testis"]

tau = compute_tau(median_tpm_filtered)
tau_df = tau.reset_index().rename(columns={"index": "Name"})
tau_df = tau_df.merge(
    gencode_genes[["gene_id_v47", "gene_type_grouped", "change_level"]],
    left_on="Name", right_on="gene_id_v47", how="inner",
)

tau_nt = compute_tau(median_tpm_filtered[non_testis_cols])
tau_nt.name = "tau_no_testis"

tau_plot = tau_df[tau_df["gene_type_grouped"] != "artifact"].dropna(subset=["tau"]).copy()
tau_plot["tau_category"] = tau_plot["tau"].apply(get_tau_category)
tau_plot = tau_plot.merge(tau_nt.reset_index().rename(columns={"index": "Name"}), on="Name", how="left")
tau_plot["tau_category_no_testis"] = tau_plot["tau_no_testis"].apply(get_tau_category)

tissue_specific = tau_plot[tau_plot["tau_category"] == "Tissue specific"].copy()
tpm_all = median_tpm.loc[tissue_specific["Name"]]
tpm_nt = median_tpm.loc[tissue_specific["Name"], non_testis_cols]

gene_max = pd.DataFrame({
    "max_tissue_all": tpm_all.idxmax(axis=1),
    "max_expr_all": tpm_all.max(axis=1),
    "max_tissue_no_testis": tpm_nt.idxmax(axis=1),
    "max_expr_no_testis": tpm_nt.max(axis=1),
    "gene_type_grouped": tissue_specific.set_index("Name")["gene_type_grouped"],
    "tau_category_no_testis": tissue_specific.set_index("Name")["tau_category_no_testis"],
})
gene_max["max_organ_all"] = gene_max["max_tissue_all"].map(lambda t: tissue_to_organ_map.get(t, t))
gene_max["max_organ_no_testis"] = gene_max["max_tissue_no_testis"].map(
    lambda t: tissue_to_organ_map.get(t, t)
)

# Hybrid assignment for panel c:
# - Testis-only (all-tissue peak Testis, not TS without Testis) -> Testis
# - Testis max all-tissue but TS to another tissue without Testis -> that other organ
# - Otherwise -> all-tissue peak organ
is_testis_peak = gene_max["max_organ_all"] == "Testis"
is_ts_no_testis = gene_max["tau_category_no_testis"] == "Tissue specific"
gene_max["ts_organ_batch"] = np.where(
    ~is_testis_peak,
    "non_testis_peak",
    np.where(is_ts_no_testis, "rescued_from_testis", "testis_only"),
)
gene_max["max_tissue"] = np.where(
    gene_max["ts_organ_batch"] == "rescued_from_testis",
    gene_max["max_tissue_no_testis"],
    gene_max["max_tissue_all"],
)
gene_max["max_organ"] = gene_max["max_tissue"].map(lambda t: tissue_to_organ_map.get(t, t))
gene_max["gene_type_grouped"] = pd.Categorical(
    gene_max["gene_type_grouped"], categories=tau_gene_type_order, ordered=True
)

organ_counts = gene_max.groupby(["gene_type_grouped", "max_organ"]).size().reset_index(name="n_genes")
organ_counts["total"] = organ_counts.groupby("gene_type_grouped")["n_genes"].transform("sum")
organ_counts["frac"] = organ_counts["n_genes"] / organ_counts["total"]

organ_totals = organ_counts.groupby("max_organ")["n_genes"].sum().sort_values(ascending=False)
organs_sorted = organ_totals.index.tolist()
plot_mat = organ_counts.pivot_table(
    index="gene_type_grouped", columns="max_organ", values="n_genes", fill_value=0
).reindex(index=tau_gene_type_order, columns=organs_sorted, fill_value=0)
rep_tissue_by_organ = gene_max.groupby("max_organ")["max_tissue"].apply(lambda s: s.value_counts().idxmax())
organ_colors = {organ: tissue_palette.get(rep_tissue_by_organ[organ], "#ccc") for organ in organs_sorted}

tissue_specific_counts = tissue_specific["gene_type_grouped"].value_counts().reindex(tau_gene_type_order, fill_value=0)
_batch_counts = gene_max["ts_organ_batch"].value_counts()
print(
    f"Tissue-specific genes: {len(tissue_specific):,}; organs in panel c: {len(organs_sorted)}; "
    f"batches: non_testis_peak={int(_batch_counts.get('non_testis_peak', 0)):,}, "
    f"rescued_from_testis={int(_batch_counts.get('rescued_from_testis', 0)):,}, "
    f"testis_only={int(_batch_counts.get('testis_only', 0)):,}"
)


### Tissue-specific odds ratios (expression-adjusted)


In [ ]:
_tau_expr = tau_df.copy()
_tau_expr["max_tpm"] = median_tpm.max(axis=1).reindex(_tau_expr["Name"]).values
_gene_types_strat = ["novel_lncRNA", "existing_lncRNA", "protein_coding", "pseudogene", "sRNA"]
_tau_strat = _tau_expr[
    _tau_expr["gene_type_grouped"].isin(_gene_types_strat) & (_tau_expr["max_tpm"] > 0)
].dropna(subset=["tau"]).copy()
_tau_strat["log_max_tpm"] = np.log10(_tau_strat["max_tpm"] + 1)
_tau_strat["tissue_specific"] = (_tau_strat["tau"] > 0.85).astype(int)
_tau_strat["gene_type_grouped"] = pd.Categorical(
    _tau_strat["gene_type_grouped"], categories=_gene_types_strat, ordered=True
)
_model_logit = smf.logit(
    'tissue_specific ~ log_max_tpm + C(gene_type_grouped, Treatment(reference="novel_lncRNA"))',
    data=_tau_strat,
).fit(disp=False)
_reference_cat = "novel_lncRNA"
cat_prefix = 'C(gene_type_grouped, Treatment(reference="novel_lncRNA"))[T.'
odds_data = []
for gt in tau_gene_type_order:
    if gt == _reference_cat:
        or_val, ci_low, ci_high, pvalue = 1.0, 1.0, 1.0, np.nan
    else:
        param_name = f"{cat_prefix}{gt}]"
        coef = _model_logit.params[param_name]
        ci = _model_logit.conf_int().loc[param_name]
        pvalue = _model_logit.pvalues[param_name]
        or_val, ci_low, ci_high = np.exp(coef), np.exp(ci[0]), np.exp(ci[1])
    odds_data.append({"gene_type": gt, "OR": or_val, "CI_low": ci_low, "CI_high": ci_high, "pvalue": pvalue})
odds_df = pd.DataFrame(odds_data)
print("Tissue-specific odds ratios ready")


### Tissue-concordant lncRNAs (macaque and/or marmoset)

In [ ]:
upset_set_order = ["macaque", "marmoset", "both", "neither"]
upset_plot_order = ["macaque", "marmoset", "both"]
upset_set_pretty = {
    "macaque": "Macaque only",
    "marmoset": "Marmoset only",
    "both": "Both",
    "neither": "Neither",
}


def upset_set_label(mac_ok, mar_ok):
    if mac_ok and mar_ok:
        return "both"
    if mac_ok:
        return "macaque"
    if mar_ok:
        return "marmoset"
    return "neither"


_lnc_ts = lnc_meta.loc[lnc_meta["is_tissue_specific"]].copy()
_lnc_ts = _lnc_ts.merge(
    gencode_genes[["gene_id_v47", "gene_type_grouped"]],
    left_on="gene_id",
    right_on="gene_id_v47",
    how="inner",
)
_lnc_ts = _lnc_ts.loc[_lnc_ts["gene_type_grouped"].isin(lnc_split_order)].copy()
_lnc_ts["max_tissue"] = _lnc_ts["peak_tissue"]
_lnc_ts["max_organ"] = _lnc_ts["max_tissue"].map(lambda t: tissue_to_organ_map.get(t, t))
_lnc_ts["has_ortholog"] = _lnc_ts["macaque_ortholog"] | _lnc_ts["marmoset_ortholog"]
_lnc_ts["nhp_match"] = _lnc_ts["macaque_tissue_concordant"] | _lnc_ts["marmoset_tissue_concordant"]
_lnc_ts["upset_set"] = [
    upset_set_label(mac, mar)
    for mac, mar in zip(_lnc_ts["macaque_tissue_concordant"], _lnc_ts["marmoset_tissue_concordant"])
]
_lnc_ts["upset_set"] = pd.Categorical(_lnc_ts["upset_set"], categories=upset_set_order, ordered=True)
_lnc_ts["gene_type_grouped"] = pd.Categorical(
    _lnc_ts["gene_type_grouped"], categories=lnc_split_order, ordered=True
)

upset_organ_counts = (
    _lnc_ts.groupby(["upset_set", "gene_type_grouped", "max_organ"], observed=False)
    .size()
    .reset_index(name="n_genes")
)

nhp_matched_ts = _lnc_ts.loc[_lnc_ts["nhp_match"]].copy()
_total_lnc_ts = len(_lnc_ts)
_total_with_orth = int(_lnc_ts["has_ortholog"].sum())
_total_nhp_matched = len(nhp_matched_ts)
_upset_n = _lnc_ts["upset_set"].value_counts().reindex(upset_set_order, fill_value=0)
print(
    f"Tissue-specific lncRNAs: {_total_lnc_ts:,}; with ≥1 ortholog: {_total_with_orth:,}; "
    f"NHP peak match: {_total_nhp_matched:,}"
)
print(
    "Exclusive sets (Macaque only / Marmoset only / Both / neither): "
    + ", ".join([f"{k}={int(v):,}" for k, v in _upset_n.items()])
)

### Example gene TPM


In [ ]:
plot_gene_id = gencode_genes.loc[
    gencode_genes["gene_id_v47"].str.startswith(example_gene),
    "gene_id_v47",
].iloc[0]
plot_gene_name = lncrna_names.get(plot_gene_id, example_gene)
print(f"example gene: {plot_gene_name} ({plot_gene_id})")

_example_tau = tau_df.loc[tau_df["Name"] == plot_gene_id, "tau"].iloc[0]
_example_tpm = median_tpm.loc[plot_gene_id]
_example_max_tissue = _example_tpm.idxmax()
_example_max_tpm = _example_tpm.max()
print(
    f"GTEx tau={_example_tau:.4f} ({get_tau_category(_example_tau)}); "
    f"max tissue: {_example_max_tissue} (median TPM={_example_max_tpm:.3f})"
)


def load_sample_tissue_map(meta, sample_col, tissue_col, exclude_col=None):
    meta = meta.copy()
    if exclude_col is not None and exclude_col in meta.columns:
        meta = meta[meta[exclude_col] != "EXCLUDE"]
    return meta.drop_duplicates(sample_col).set_index(sample_col)[tissue_col]


def tpm_to_long(tpm_df, gene_ids, sample_to_tissue, dataset, human_gene_map=None, id_col="Name"):
    cols = ["dataset", "gene_id", "human_gene_id", "tissue", "tpm"]
    df = tpm_df[tpm_df[id_col].isin(gene_ids)]
    if df.empty:
        return pd.DataFrame(columns=cols)
    sample_cols = [c for c in df.columns if c not in {"Name", "Description", "Gene"}]
    long = df.melt(id_vars=[id_col], value_vars=sample_cols, var_name="sample_id", value_name="tpm")
    long = long.rename(columns={id_col: "gene_id"})
    long["tissue"] = long["sample_id"].map(sample_to_tissue)
    long = long.dropna(subset=["tissue"])
    if human_gene_map is None:
        long["human_gene_id"] = long["gene_id"]
    else:
        long["human_gene_id"] = long["gene_id"].map(human_gene_map)
    long["dataset"] = dataset
    return long[cols]


gtex_sample_map = load_sample_tissue_map(gtex_sample_meta, "SAMPID", "SMTSD")
dgtex_sample_map = load_sample_tissue_map(
    dgtex_meta, "Sample ID (Data Release Official ID)", "Tissue Site Detail", "EXCLUDE from Analysis Freeze"
)
macaque_sample_map = load_sample_tissue_map(
    macaque_meta, "Sample ID (Data Release Official ID)", "Tissue Site Detail"
)
marmoset_sample_map = load_sample_tissue_map(
    marmoset_meta, "Sample ID (Data Release Official ID)", "Tissue Site Detail"
)

macaque_id = orth.loc[plot_gene_id, "gene_id.macaque"] if plot_gene_id in orth.index else None
marmoset_id = orth.loc[plot_gene_id, "gene_id.marmoset"] if plot_gene_id in orth.index else None
print(f"macaque ortholog: {macaque_id}; marmoset ortholog: {marmoset_id}")

human_ids = [plot_gene_id]
mac_ids = [macaque_id] if pd.notna(macaque_id) else []
mar_ids = [marmoset_id] if pd.notna(marmoset_id) else []
macaque_to_human = {macaque_id: plot_gene_id} if pd.notna(macaque_id) else {}
marmoset_to_human = {marmoset_id: plot_gene_id} if pd.notna(marmoset_id) else {}

bulk_tpm_long = pd.concat([
    tpm_to_long(gtex_tpm, human_ids, gtex_sample_map, "GTEx"),
    tpm_to_long(dgtex_tpm, human_ids, dgtex_sample_map, "dGTEx"),
    tpm_to_long(macaque_tpm, mac_ids, macaque_sample_map, "Macaque", macaque_to_human),
    tpm_to_long(marmoset_tpm, mar_ids, marmoset_sample_map, "Marmoset", marmoset_to_human),
], ignore_index=True)
print(f"bulk TPM: {len(bulk_tpm_long):,} sample rows for {plot_gene_id}")

bulk_site_long = bulk_tpm_long.copy()
for dataset, tissue_map in detail_to_site_by_dataset.items():
    mask = bulk_site_long["dataset"] == dataset
    bulk_site_long.loc[mask, "tissue_site"] = bulk_site_long.loc[mask, "tissue"].map(
        lambda t: tissue_map.get(t, t)
    )

# Genomic context for panel-e inset (gene + exon models near example lncRNA)
_split_by_v47 = gencode_genes.set_index("gene_id_v47")["gene_type_grouped"]
_stable_split = gencode_genes.set_index("gene_id_stable")["gene_type_grouped"]
_focal_gene = gencode_annot.filter(
    pc.and_(pc.equal(gencode_annot["Feature"], "gene"), pc.equal(gencode_annot["gene_id"], plot_gene_id))
).to_pandas().iloc[0]
_example_chrom = _focal_gene["Chromosome"]
_example_gene_start = int(_focal_gene["Start"])
_example_gene_end = int(_focal_gene["End"])
# Compact window: show nearest up/downstream genes without expanding to full megabase spans
_chrom_genes = gencode_annot.filter(
    pc.and_(pc.equal(gencode_annot["Chromosome"], _example_chrom), pc.equal(gencode_annot["Feature"], "gene"))
).to_pandas()
_chrom_other = _chrom_genes[_chrom_genes["gene_id"] != plot_gene_id]
_up = _chrom_other[_chrom_other["End"] < _example_gene_start].sort_values("End", ascending=False)
_down = _chrom_other[_chrom_other["Start"] > _example_gene_end].sort_values("Start", ascending=True)
# Tight flank; only pull in neighbors that abut/near the focal gene (skip distant tiny genes)
_flank = example_flank_bp
_neighbor_show = example_neighbor_show_bp
_max_neighbor_gap = example_max_neighbor_gap_bp
_example_win_lo = _example_gene_start - _flank
_example_win_hi = _example_gene_end + _flank
if len(_up):
    _u = _up.iloc[0]
    if _example_gene_start - int(_u["End"]) <= _max_neighbor_gap:
        _example_win_lo = min(_example_win_lo, max(int(_u["Start"]), int(_u["End"]) - _neighbor_show))
if len(_down):
    _d = _down.iloc[0]
    if int(_d["Start"]) - _example_gene_end <= _max_neighbor_gap:
        _example_win_hi = max(_example_win_hi, min(int(_d["End"]), int(_d["Start"]) + _neighbor_show))

_win_gene_mask = pc.and_(
    pc.and_(pc.equal(gencode_annot["Chromosome"], _example_chrom), pc.equal(gencode_annot["Feature"], "gene")),
    pc.and_(pc.less_equal(gencode_annot["Start"], _example_win_hi), pc.greater_equal(gencode_annot["End"], _example_win_lo)),
)
example_ctx_genes = gencode_annot.filter(_win_gene_mask).to_pandas().sort_values("Start").reset_index(drop=True)
_win_exon_mask = pc.and_(
    pc.and_(pc.equal(gencode_annot["Chromosome"], _example_chrom), pc.equal(gencode_annot["Feature"], "exon")),
    pc.and_(pc.less_equal(gencode_annot["Start"], _example_win_hi), pc.greater_equal(gencode_annot["End"], _example_win_lo)),
)
_example_exons = gencode_annot.filter(_win_exon_mask).to_pandas()
_rep_tx = (
    _example_exons.assign(exon_len=lambda d: d["End"] - d["Start"])
    .groupby(["gene_id", "transcript_id"], as_index=False)["exon_len"]
    .sum()
    .sort_values("exon_len", ascending=False)
    .groupby("gene_id", as_index=False)
    .first()
)
example_ctx_exons = _example_exons.merge(_rep_tx[["gene_id", "transcript_id"]], on=["gene_id", "transcript_id"])

def _biotype_for_gid(gid, gtf_type):
    if gid in _split_by_v47.index:
        return _split_by_v47.loc[gid]
    stable = str(gid).split(".")[0]
    if stable in _stable_split.index:
        return _stable_split.loc[stable]
    if gtf_type == "protein_coding":
        return "protein_coding"
    if "pseudo" in str(gtf_type):
        return "pseudogene"
    return "existing_lncRNA"

example_ctx_genes["biotype"] = [
    _biotype_for_gid(gid, gt) for gid, gt in zip(example_ctx_genes["gene_id"], example_ctx_genes["gene_type"])
]
del gencode_annot
print(
    f"genomic context: {_example_chrom}:{_example_win_lo}-{_example_win_hi}; "
    f"{len(example_ctx_genes)} genes, {len(example_ctx_exons)} exons"
)
print(example_ctx_genes[["gene_name", "gene_type", "biotype", "Start", "End", "Strand"]].to_string(index=False))

## Make figure


In [ ]:
fig = plt.figure(figsize=(fig_width, fig_height), dpi=fig_dpi)
gs_main = fig.add_gridspec(2, 1, height_ratios=[0.42, 0.58], hspace=0.55)

# Shared column geometry so a|b matches d|e and b's left spine aligns with e's
_col_ratios = [0.5, 1.0, 0.5]
_wspace = 0.45

# Top: [a expr | b τ-ECDF | c organ TS]
gs_top = gs_main[0].subgridspec(1, 3, width_ratios=_col_ratios, wspace=_wspace)
ax_expr = fig.add_subplot(gs_top[0])
ax_tau = fig.add_subplot(gs_top[1])
ax_org = fig.add_subplot(gs_top[2])

# Bottom: d under a; e spans b+c columns; dataset strip only under e
gs_bottom = gs_main[1].subgridspec(
    2, 3, height_ratios=[1, 0.055], width_ratios=_col_ratios,
    hspace=0.12, wspace=_wspace,
)
ax_upset = fig.add_subplot(gs_bottom[0, 0])
ax_example = fig.add_subplot(gs_bottom[0, 1:])
ax_example_ann = fig.add_subplot(gs_bottom[1, 1:], sharex=ax_example)

# Left→right biotype order for vertical panels a/c
bar_xticks = np.arange(len(bar_gene_type_order))
bar_xticklabels = [gene_type_pretty.get(gt, gt) for gt in bar_gene_type_order]

# --- a: % expressed vs intergenic (vertical) ---
sns.stripplot(
    data=tissue_summary, x="gene_type_grouped", y="pct",
    order=bar_gene_type_order, hue="tissue", palette=tissue_palette,
    dodge=False, size=1.2, linewidth=0, alpha=0.5, zorder=2, ax=ax_expr, orient="v",
    legend=False, jitter=0.2,
)
for gt, pct_any, _ in _hollow_rows:
    x = bar_gene_type_order.index(gt)
    color = gene_type_palette[gt]
    rgb = matplotlib.colors.to_rgb(color)
    ax_expr.bar(
        x, pct_any, width=0.8,
        edgecolor=color, facecolor=(*rgb, 0.08), linewidth=0.8, zorder=1,
    )
    ax_expr.text(
        x, pct_any + 1.4, f"{pct_any:.1f}%", va="bottom", ha="center",
        fontsize=4, color="black", clip_on=False,
    )
ax_expr.set_xticks(bar_xticks)
ax_expr.set_xticklabels(bar_xticklabels, rotation=45, ha="right")
ax_expr.set_xlabel("")
ax_expr.set_ylabel("% expressed vs intergenic")
ax_expr.set_ylim(-1, 110)
ax_expr.set_xlim(-0.5, len(bar_gene_type_order) - 0.5)
ax_expr.tick_params(axis="x", bottom=False, length=0)
ax_expr.spines[["top", "right", "bottom"]].set_visible(False)

# --- b: reverse ECDF of τ by biotype ---
for gt in bar_gene_type_order:
    gt_data = tau_plot[tau_plot["gene_type_grouped"] == gt]["tau"].dropna()
    if len(gt_data) == 0:
        continue
    sorted_tau = np.sort(gt_data.values)
    x = np.insert(sorted_tau, 0, 0.0)
    y = 100.0 * (len(sorted_tau) - np.arange(0, len(sorted_tau) + 1)) / len(sorted_tau)
    ax_tau.plot(x, y, color=gene_type_palette.get(gt), lw=1.2, zorder=2)

for _cut in (tau_housekeeping, tau_tissue_specific):
    ax_tau.axvline(_cut, color="0.55", ls=":", lw=0.7, zorder=1)
for _left, _right, _label in (
    (0.0, tau_housekeeping, "Housekeeping"),
    (tau_housekeeping, tau_tissue_specific, "Midrange"),
    (tau_tissue_specific, 1.0, "Tissue specific"),
):
    ax_tau.text(
        (_left + _right) / 2, 1.02, _label,
        transform=ax_tau.get_xaxis_transform(),
        ha="center", va="bottom", fontsize=4, color="k", clip_on=False,
    )
ax_tau.set_xlim(0, 1.01)
ax_tau.set_ylim(0, 105)
ax_tau.set_xlabel("Tissue specificity (τ)")
ax_tau.set_ylabel("% of genes ≥ τ")
ax_tau.set_xticks([0, tau_housekeeping, 0.5, tau_tissue_specific, 1])
ax_tau.set_xticklabels(["0", "0.15", "0.5", "0.85", "1"])
ax_tau.spines[["top", "right"]].set_visible(False)

# Panel b inset: expression-adjusted TS ORs (novel lncRNA reference; doubles as legend)
def _sig_stars(p):
    if not np.isfinite(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""

_ins_order = ["novel_lncRNA", "existing_lncRNA", "protein_coding", "pseudogene"]
_ins_df = odds_df.set_index("gene_type").loc[_ins_order].reset_index()
ax_tau_ins = ax_tau.inset_axes([0.18, 0.14, 0.32, 0.24])
ax_tau_ins.set_facecolor("white")
ax_tau_ins.patch.set_alpha(0.92)
ax_tau_ins.axvline(0.0, color="0.5", ls="--", lw=0.5, zorder=0)
for i, row in _ins_df.iterrows():
    gt = row["gene_type"]
    color = gene_type_palette[gt]
    or_log2 = np.log2(row["OR"]) if row["OR"] > 0 else np.nan
    lo = np.log2(row["CI_low"]) if row["CI_low"] > 0 else np.nan
    hi = np.log2(row["CI_high"]) if row["CI_high"] > 0 else np.nan
    y = len(_ins_order) - 1 - i
    ax_tau_ins.plot([lo, hi], [y, y], color=color, lw=0.65, zorder=1)
    is_ref = gt == "novel_lncRNA"
    ax_tau_ins.scatter(
        or_log2, y, color=color, s=6, marker=("D" if is_ref else "o"),
        edgecolor="k", linewidths=0.2, zorder=3,
    )
    if is_ref and np.isfinite(or_log2):
        ax_tau_ins.text(
            or_log2 - 0.12, y, "Ref",
            va="center", ha="right", fontsize=3, style="italic", color="0.2",
        )
    else:
        stars = _sig_stars(row["pvalue"])
        if stars and np.isfinite(or_log2):
            ax_tau_ins.text(or_log2, y + 0.16, stars, va="bottom", ha="center", fontsize=3)
ax_tau_ins.set_yticks(range(len(_ins_order)))
ax_tau_ins.set_yticklabels(
    [gene_type_pretty[g] for g in _ins_order[::-1]], fontsize=3
)
ax_tau_ins.set_ylim(-0.55, len(_ins_order) - 0.45)
ax_tau_ins.set_xticks([-2, -1, 0])
ax_tau_ins.set_xticklabels(["1/4", "1/2", "1"], fontsize=3)
ax_tau_ins.set_xlabel("OR T.S. ~ expr.", fontsize=3.5, labelpad=0.5)
ax_tau_ins.tick_params(axis="both", length=1.0, pad=0.6)
for _sp in ax_tau_ins.spines.values():
    _sp.set_visible(True)
    _sp.set_linewidth(0.5)
    _sp.set_color("0.2")

# --- c: tissue-specific gene counts by peak organ (vertical stacked) ---
plot_mat_v = plot_mat.reindex(index=bar_gene_type_order, fill_value=0)
prev_bottom = np.zeros(len(bar_gene_type_order))
bar_handles = []
for organ in organs_sorted:
    values = plot_mat_v[organ].values.astype(float)
    bars = ax_org.bar(
        bar_xticks, values, bottom=prev_bottom, width=0.8,
        color=organ_colors.get(organ, "#ccc"), edgecolor="none",
        label=organ, alpha=0.8, zorder=2,
    )
    bar_handles.append(bars[0])
    prev_bottom += values
for i, gt in enumerate(bar_gene_type_order):
    ax_org.bar(
        i, prev_bottom[i], bottom=0, width=0.8,
        facecolor="none", edgecolor=gene_type_palette[gt], linewidth=0.8, zorder=3,
    )
    ax_org.text(
        i, prev_bottom[i], f"{int(tissue_specific_counts.get(gt, 0)):,}",
        va="bottom", ha="center", fontsize=4, color="black", clip_on=False,
    )
ax_org.set_xticks(bar_xticks)
ax_org.set_xticklabels(bar_xticklabels, rotation=45, ha="right")
ax_org.set_xlabel("")
ax_org.set_ylabel("Tissue-specific genes")
ax_org.set_ylim(0, (max(prev_bottom) * 1.12) if len(prev_bottom) else 1)
ax_org.set_xlim(-0.5, len(bar_gene_type_order) - 0.5)
ax_org.tick_params(axis="x", bottom=False, length=0)
ax_org.spines[["top", "right", "bottom"]].set_visible(False)
ax_org.set_yticks([0, 2500, 5000, 7500, 10000])
ax_org.set_yticklabels([0, '2.5k', '5k', '7.5k', '10k'])

# --- d: UpSet-style NHP exact/Uberon-matched same-tissue counts ---
bar_width = 0.38
pair_gap = 0.08
group_pitch = 1.15
group_centers = []
xmax_count = 0
outline_handles = {}
for i, upset_set in enumerate(upset_plot_order):
    pair_left = i * group_pitch
    # Midpoint between the two centered bars in the pair
    group_centers.append(pair_left + (bar_width + pair_gap) / 2)
    for j, gt in enumerate(lnc_split_order):
        x = pair_left + j * (bar_width + pair_gap)
        bottom = 0.0
        sub = upset_organ_counts[
            (upset_organ_counts["upset_set"] == upset_set)
            & (upset_organ_counts["gene_type_grouped"] == gt)
        ]
        height_by_organ = sub.set_index("max_organ")["n_genes"].reindex(organs_sorted, fill_value=0)
        for organ, h in height_by_organ.items():
            h = float(h)
            if h <= 0:
                continue
            ax_upset.bar(
                x, h, bottom=bottom, width=bar_width,
                color=organ_colors.get(organ, "#ccc"), edgecolor="none",
                alpha=0.85, zorder=2,
            )
            bottom += h
        # overall outline encodes novel vs existing
        outline = ax_upset.bar(
            x, bottom if bottom > 0 else 0, bottom=0, width=bar_width,
            facecolor="none", edgecolor=gene_type_palette[gt], linewidth=0.9, zorder=3,
        )
        if gt not in outline_handles:
            outline_handles[gt] = outline
        if bottom > 0:
            ax_upset.text(
                x, bottom, f"{int(bottom):,}",
                ha="center", va="bottom", fontsize=5, clip_on=False,
            )
        xmax_count = max(xmax_count, bottom)

ax_upset.set_xlim(-0.35, (len(upset_plot_order) - 1) * group_pitch + 2 * bar_width + pair_gap + 0.35)
ax_upset.set_ylim(0, xmax_count * 1.12 if xmax_count else 1)
ax_upset.set_xticks(group_centers)
ax_upset.set_xticklabels([upset_set_pretty[s] for s in upset_plot_order], rotation=45)
ax_upset.set_ylabel("Tissue-concordant lncRNAs")
ax_upset.spines[["top", "right"]].set_visible(False)
ax_upset.legend(
    handles=[
        Patch(
            facecolor="white", edgecolor=gene_type_palette[gt], linewidth=0.9,
            label=gene_type_pretty.get(gt, gt),
        )
        for gt in lnc_split_order
    ],
    loc="upper right",
    ncol=1,
    frameon=False,
    fontsize=4,
    handlelength=1.1,
    handletextpad=0.35,
    columnspacing=0.8,
    borderaxespad=0.1,
)

def site_dict_lookup(d, site, default=None):
    if site in d:
        return d[site]
    site_lower = site.lower()
    for key, value in d.items():
        if key.lower() == site_lower:
            return value
    return default

dataset_ann_palette = {
    "GTEx": "#222222",
    "dGTEx": "#666666",
    "Macaque": "#AAAAAA",
    "Marmoset": "#DDDDDD",
}

gene_sites = bulk_site_long.query("human_gene_id == @plot_gene_id")
# drop tissue sites with fewer than 3 contributing subjects/samples
gene_sites = gene_sites.groupby(["dataset", "tissue_site"]).filter(lambda g: len(g) >= example_min_samples_per_site)
panel_specs = [
    ("GTEx", sorted(gene_sites.query('dataset == "GTEx"')["tissue_site"].unique().tolist())),
    ("dGTEx", sorted(gene_sites.query('dataset == "dGTEx"')["tissue_site"].unique().tolist())),
    ("Macaque", sorted(gene_sites.query('dataset == "Macaque"')["tissue_site"].unique().tolist())),
    ("Marmoset", sorted(gene_sites.query('dataset == "Marmoset"')["tissue_site"].unique().tolist())),
]

x_order, x_labels, x_datasets, example_tissue_palette = [], [], [], {}
dataset_spans = {}
for dataset, tissue_sites in panel_specs:
    if not tissue_sites:
        continue
    start = len(x_order)
    abbr_map = site_abbrv_by_dataset[dataset]
    site_pal = site_palette_by_dataset.get(dataset, {})
    for site in tissue_sites:
        key = f"{dataset}::{site}"
        x_order.append(key)
        x_labels.append(site_dict_lookup(abbr_map, site, site))
        x_datasets.append(dataset)
        example_tissue_palette[key] = site_dict_lookup(site_pal, site, "#cccccc")
    dataset_spans[dataset] = (start, len(x_order) - 1)

plot_df = gene_sites.copy()
plot_df["x_key"] = plot_df["dataset"] + "::" + plot_df["tissue_site"].astype(str)
plot_df = plot_df[plot_df["x_key"].isin(x_order)]
_med_max = float(plot_df.groupby("x_key")["tpm"].median().max()) if len(plot_df) else 1.0
_tpm_clip = max(_med_max * 2.0, 1.0)
plot_df["tpm"] = plot_df["tpm"].clip(upper=_tpm_clip)

sns.boxplot(
    data=plot_df,
    x="x_key",
    y="tpm",
    order=x_order,
    hue="x_key",
    hue_order=x_order,
    palette=example_tissue_palette,
    dodge=False,
    ax=ax_example,
    linewidth=0.4,
    saturation=1.0,
    fliersize=1.2,
    width=0.7,
    flierprops=dict(marker="o", markersize=1.2, markeredgewidth=0, linestyle="none"),
)
flier_lines = [
    ln for ln in ax_example.get_lines()
    if ln.get_linestyle() == "None" and ln.get_marker() == "o"
]
for flier in flier_lines:
    xdata = flier.get_xdata()
    if len(xdata) == 0:
        continue
    xi = int(round(xdata[0]))
    if 0 <= xi < len(x_order):
        color = example_tissue_palette[x_order[xi]]
        flier.set_markerfacecolor(color)
        flier.set_markeredgecolor(color)

for dataset, (start, end) in dataset_spans.items():
    if start > 0:
        ax_example.axvline(start - 0.5, color="k", lw=0.5, zorder=0, linestyle="--")

ax_example.set_xlim(-0.5, len(x_order) - 0.5)
ax_example.set_ylim(bottom=0)
ax_example.set_ylabel(f"{plot_gene_name} TPM")
ax_example.set_xlabel("")
ax_example.tick_params(axis="x", labelbottom=False, length=0)
ax_example.spines[["top", "right"]].set_visible(False)
if ax_example.get_legend() is not None:
    ax_example.get_legend().remove()

for i, d in enumerate(x_datasets):
    ax_example_ann.add_patch(plt.Rectangle(
        (i - 0.5, 0), 1, 1,
        facecolor=dataset_ann_palette[d], edgecolor="none", linewidth=0,
        transform=ax_example_ann.transData, clip_on=False,
    ))
ax_example_ann.set_xlim(-0.5, len(x_order) - 0.5)
ax_example_ann.set_ylim(0, 1)
ax_example_ann.set_yticks([])
ax_example_ann.set_xticks([])
ax_example_ann.set_xlabel("")
for spine in ax_example_ann.spines.values():
    spine.set_visible(False)
# dataset labels under the colored bars
for dataset, (start, end) in dataset_spans.items():
    mid = (start + end) / 2
    ax_example_ann.text(
        mid, -0.55, dataset, ha="center", va="top", fontsize=4, clip_on=False,
    )

# Organ legend on panel e (far right, overlapping axes whitespace — not a separate axes)
legend_handles, legend_labels, _seen = [], [], set()
for key in x_order:
    site = key.split("::", 1)[1]
    if str(site).lower() in _seen:
        continue
    legend_handles.append(Patch(facecolor=example_tissue_palette[key], edgecolor="none"))
    legend_labels.append(site)
    _seen.add(str(site).lower())
for organ, handle in zip(organs_sorted, bar_handles):
    if str(organ).lower() in _seen:
        continue
    legend_handles.append(handle)
    legend_labels.append(organ)
    _seen.add(str(organ).lower())
ax_example.legend(
    handles=legend_handles,
    labels=legend_labels,
    title="Organ",
    loc="center left",
    bbox_to_anchor=(.85, 0.52),
    frameon=True,
    fancybox=False,
    edgecolor="none",
    facecolor="white",
    framealpha=0.9,
    borderpad=0.3,
    labelspacing=0.25,
    columnspacing=0.6,
    handlelength=0.9,
    handletextpad=0.35,
    ncol=2,
    fontsize=3.2,
    title_fontsize=4,
)

# --- e inset: genomic context (does not alter GridSpec / other axes) ---
ax_ins = ax_example.inset_axes([0.18, 0.66, 0.50, 0.3])
ax_ins.set_facecolor("white")
for sp in ax_ins.spines.values():
    sp.set_visible(True)
    sp.set_linewidth(0.7)
    sp.set_color("0.15")

_y_plus, _y_minus = 0.52, 0.20
for _y in (_y_plus, _y_minus):
    ax_ins.axhline(_y, color="0.8", lw=0.45, zorder=0)
_strand_trans = blended_transform_factory(ax_ins.transAxes, ax_ins.transData)
ax_ins.text(
    0.985, _y_plus + 0.06, "+", transform=_strand_trans,
    ha="right", va="bottom", fontsize=5, color="0.3", clip_on=False,
)
ax_ins.text(
    0.985, _y_minus + 0.06, "−", transform=_strand_trans,
    ha="right", va="bottom", fontsize=5, color="0.3", clip_on=False,
)

def _draw_gene_model(ax, gene_row, exons, y, win_lo, win_hi, label_y):
    gid = gene_row["gene_id"]
    is_focal = gid == plot_gene_id
    color = gene_type_palette.get(gene_row["biotype"], "#666666")
    g0 = max(int(gene_row["Start"]), win_lo)
    g1 = min(int(gene_row["End"]), win_hi)
    ax.plot([g0, g1], [y, y], color=color, lw=1.4 if is_focal else 0.8, solid_capstyle="butt", zorder=1)
    gap = 0.01 * (win_hi - win_lo)
    if int(gene_row["Start"]) < win_lo:
        for dx in (0.0, gap * 0.45):
            ax.plot([win_lo + dx, win_lo + dx + gap * 0.25], [y - 0.06, y + 0.06], color=color, lw=0.6, zorder=3)
    if int(gene_row["End"]) > win_hi:
        for dx in (0.0, gap * 0.45):
            ax.plot([win_hi - gap - dx, win_hi - gap * 0.75 - dx], [y - 0.06, y + 0.06], color=color, lw=0.6, zorder=3)
    h = 0.22 if is_focal else 0.14
    min_exon_w = 0.005 * (win_hi - win_lo)  # keep short exons visible when zoomed out
    gene_exons = exons[exons["gene_id"] == gid].sort_values("Start")
    for _, ex in gene_exons.iterrows():
        x0 = max(int(ex["Start"]), win_lo)
        x1 = min(int(ex["End"]), win_hi)
        if x1 <= x0:
            continue
        w = max(x1 - x0, min_exon_w)
        ax.add_patch(
            plt.Rectangle(
                (x0, y - h / 2), w, h,
                facecolor=color, edgecolor="none", zorder=2, linewidth=0,
            )
        )
    # Labels sit above gene midpoints; staggered up when same-strand names would collide
    label = gene_row["gene_name"]
    ax.text(
        0.5 * (g0 + g1), label_y, label,
        ha="center", va="bottom",
        fontsize=4, color=color,
        style="italic", clip_on=False, zorder=4,
    )

# Stagger same-strand labels vertically when gene midpoints are close
_label_slots = {"+": [], "-": []}
_min_label_sep = 0.12 * (_example_win_hi - _example_win_lo)
for _, grow in example_ctx_genes.sort_values("Start").iterrows():
    _strand = grow["Strand"]
    _y = _y_plus if _strand == "+" else _y_minus
    _g0 = max(int(grow["Start"]), _example_win_lo)
    _g1 = min(int(grow["End"]), _example_win_hi)
    _mid = 0.5 * (_g0 + _g1)
    _slot = 0
    for _prev_mid, _prev_slot in _label_slots[_strand]:
        if abs(_mid - _prev_mid) < _min_label_sep:
            _slot = max(_slot, _prev_slot + 1)
    _label_slots[_strand].append((_mid, _slot))
    _draw_gene_model(
        ax_ins, grow, example_ctx_exons, _y, _example_win_lo, _example_win_hi,
        label_y=_y + 0.1 + _slot * 0.12,
    )

ax_ins.set_xlim(_example_win_lo, _example_win_hi)
ax_ins.set_ylim(0.0, 1.0)
ax_ins.set_yticks([])
# top spine: ticks face into the box; labels sit just below ticks (inside)
_win_width = _example_win_hi - _example_win_lo
_tick_step = 10_000 if _win_width < 80_000 else (25_000 if _win_width < 200_000 else 50_000)
_tick_start = int(np.ceil(_example_win_lo / _tick_step) * _tick_step)
_xticks = list(range(_tick_start, int(_example_win_hi) + 1, _tick_step))
if len(_xticks) > 5:
    _tick_step *= 2
    _tick_start = int(np.ceil(_example_win_lo / _tick_step) * _tick_step)
    _xticks = list(range(_tick_start, int(_example_win_hi) + 1, _tick_step))
if len(_xticks) < 2:
    _xticks = [_example_win_lo, _example_win_hi]
ax_ins.xaxis.tick_top()
ax_ins.set_xticks(_xticks)
ax_ins.set_xticklabels([])  # draw labels inside manually
ax_ins.tick_params(
    axis="x", which="major", top=True, bottom=False,
    labeltop=False, labelbottom=False, direction="in",
    length=2.5, width=0.6, colors="0.15",
)
# labels sit below the inward ticks (not overlapping)
for _x in _xticks:
    ax_ins.text(
        _x, 0.88, f"{_x:,}", ha="center", va="top", fontsize=3.5, color="0.15",
        clip_on=True, zorder=5,
    )
ax_ins.text(
    1.0, 1.02, _example_chrom, transform=ax_ins.transAxes,
    ha="right", va="bottom", fontsize=4.5, color="0.2", clip_on=False,
)
ax_ins.patch.set_alpha(0.96)

_inv = fig.transFigure.inverted()
for ax, label, x_off in [
    (ax_expr, "a", -0.18),
    (ax_tau, "b", -0.12),
    (ax_org, "c", -0.18),
    (ax_upset, "d", -0.15),
    (ax_example, "e", -0.08),
]:
    _x, _y = _inv.transform(ax.transAxes.transform((x_off, 1.05)))
    fig.text(_x, _y, label, fontweight="bold", va="bottom", ha="right", fontsize=9)
plt.show()


## Save figure


In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_pdf, bbox_inches="tight")
fig.savefig(fig_png, bbox_inches="tight")
print(f"Wrote {fig_pdf}")
print(f"Wrote {fig_png}")


## Print stats


In [ ]:
# Pooled ∪ any of 54 bulk sites (cell lines included), % of lncRNAs
for gt in ["novel_lncRNA", "existing_lncRNA"]:
    n_expr, n_total = expr_any_counts[gt]
    print(f"{gene_type_pretty[gt]} expressed (pooled ∪ tissue): {n_expr:,} / {n_total:,} ({100 * n_expr / n_total:.1f}%)")

# Median % of lncRNAs expressed per tissue (54 bulk sites)
for gt in ["novel_lncRNA", "existing_lncRNA"]:
    med_pct = tissue_summary.loc[tissue_summary["gene_type_grouped"] == gt, "pct"].median()
    print(f"{gene_type_pretty[gt]} median % expressed per tissue: {med_pct:.1f}%")

# Expression-adjusted tissue-specific OR (Results: novel vs existing; model ref = novel)
_ex = odds_df.loc[odds_df["gene_type"] == "existing_lncRNA"].iloc[0]
_or_novel_vs_existing = 1.0 / _ex["OR"]
print(
    f"tissue-specific OR, novel vs existing (expression-adjusted): "
    f"{_or_novel_vs_existing:.2f}, p = {_ex['pvalue']:.1e}"
)

# Tissue-specific novel lncRNAs by peak organ (Fig 2c)
novel_ts = gene_max[gene_max["gene_type_grouped"] == "novel_lncRNA"]
n_novel_ts = len(novel_ts)
for organ in ["Testis", "Brain"]:
    n_organ = int((novel_ts["max_organ"] == organ).sum())
    print(f"tissue-specific novel lncRNAs, {organ.lower()}: {100 * n_organ / n_novel_ts:.1f}% (n={n_organ:,})")

# Tissue-concordant lncRNAs (NHP-matched peak organ in macaque and/or marmoset)
for gt in ["novel_lncRNA", "existing_lncRNA"]:
    n_match = int((nhp_matched_ts["gene_type_grouped"] == gt).sum())
    print(f"tissue-concordant {gene_type_pretty[gt]}s (macaque and/or marmoset): {n_match:,}")

# Example brain-specific tissue-concordant novel lncRNA (Fig 2e)
print(f"{example_gene} τ = {_example_tau:.3f}")
